# 01 · Data subsample + QC + normalization

**Phase 1 of the project** ([PROJECT_PLAN.md](../PROJECT_PLAN.md)).

This notebook takes the Salcher 2022 LuCA lung-cancer atlas from disk to a
working, QC-filtered, log-normalized AnnData that downstream notebooks build on.

**Pipeline**

1. Build a stratified 100K-cell subsample from `luca_core.h5ad` (the full
   ~13 GB atlas) using `src.subsample.subsample_luca`. Skipped if the
   subsample h5ad already exists.
2. Load the subsample and inspect cohort composition.
3. Visualize LuCA's pre-computed QC metrics (`n_genes_by_counts`,
   `total_counts`, `pct_counts_mito`).
4. Filter low-quality cells + drop pre-flagged doublets.
5. Normalize + log1p + flag 2000 HVGs.
6. Save to `data/processed/luca_filtered.h5ad`.

**Inputs**: `luca_core.h5ad` (downloaded per [`data/README.md`](../data/README.md)).
**Outputs**: `data/luca_subsample_100k.h5ad`, `data/processed/luca_filtered.h5ad`, QC figures in `figures/`.

## Setup

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

# Suppress noisy FutureWarnings from anndata/scanpy at import time
warnings.filterwarnings("ignore", category=FutureWarning)

# Find the repo root, in priority order:
#   1. REPO_ROOT env var (set this if Jupyter isn't launched from the repo)
#   2. Walk up from cwd looking for src/subsample.py (works when Jupyter is
#      launched from anywhere inside the repo)
#   3. Fall back to a hardcoded path you can edit below
HARDCODED_ROOT = Path(r"E:\\jobRelated\\withClaude\\resume_building\\github-portfolio\\scrnaseq-tumor-microenvironment")

def _find_root():
    env = os.environ.get("REPO_ROOT")
    if env and (Path(env) / "src" / "subsample.py").exists():
        return Path(env).resolve()
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "subsample.py").exists():
            return p.resolve()
    if (HARDCODED_ROOT / "src" / "subsample.py").exists():
        return HARDCODED_ROOT.resolve()
    return None

ROOT = _find_root()
assert ROOT is not None, (
    "Repo root not found.\n"
    "Either (a) launch Jupyter from inside the repo, "
    "(b) set the REPO_ROOT env var before launching, "
    "or (c) edit HARDCODED_ROOT above."
)
sys.path.insert(0, str(ROOT))

from src.subsample import subsample_luca
from src.data_io import load_adata, save_adata, DATA_DIR, DEFAULT_SUBSAMPLE_NAME

DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = ROOT / "figures"
PROC_DIR = ROOT / "data" / "processed"
FIG_DIR.mkdir(exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

sc.settings.verbosity = 2
sc.settings.figdir = FIG_DIR
sc.settings.set_figure_params(dpi=100, frameon=False)

print(f"scanpy {sc.__version__}")
print(f"repo:      {ROOT}")
print(f"DATA_DIR:  {DATA_DIR}")
print(f"PROC_DIR:  {PROC_DIR}")
print(f"FIG_DIR:   {FIG_DIR}")


scanpy 1.11.5
repo:      E:\jobRelated\withClaude\resume_building\github-portfolio\scrnaseq-tumor-microenvironment
DATA_DIR:  E:\jobRelated\withClaude\resume_building\github-portfolio\scrnaseq-tumor-microenvironment\data
PROC_DIR:  E:\jobRelated\withClaude\resume_building\github-portfolio\scrnaseq-tumor-microenvironment\data\processed
FIG_DIR:   E:\jobRelated\withClaude\resume_building\github-portfolio\scrnaseq-tumor-microenvironment\figures


## 1. Build the 100K stratified subsample

The full atlas (`luca_core.h5ad`, ~13 GB) holds 892K cells × 17,764 genes plus
two huge raw-count layers (`count`, `counts_length_scaled`). Loading the whole
thing into a 16 GB machine blows up RAM, so we stream-sample directly off the
HDF5 file:

- Stratify by `study × disease` (19 cohorts × 5 disease labels = 30 strata)
- Take 3,333 cells per stratum (= ~100K target)
- Slice raw counts from `/layers/count` into `.X` (the atlas's own `.X` is
  pre-normalized, but scVI in notebook 02 needs raw counts)
- Also keep the atlas's pre-normalized `.X` as `layers['normalized_atlas']`
  for future comparison

This cell skips re-subsampling if the output file already exists. To force a
rebuild, delete `data/luca_subsample_100k.h5ad` or pass a different
`n_target` / `seed`. Set the `LUCA_CORE_PATH` environment variable if your
copy of the full atlas isn't at `data/raw/luca_core.h5ad`.

In [2]:
sub_path = DATA_DIR / DEFAULT_SUBSAMPLE_NAME

if sub_path.exists():
    print(f"Subsample exists: {sub_path}")
    print(f"  size: {sub_path.stat().st_size / 1e6:.0f} MB — skipping rebuild")
else:
    luca_core_path = Path(os.environ.get(
        "LUCA_CORE_PATH",
        ROOT / "data" / "raw" / "luca_core.h5ad",
    ))
    print(f"Building subsample from: {luca_core_path}")
    if not luca_core_path.exists():
        raise FileNotFoundError(
            f"Full atlas not found at {luca_core_path}. "
            "See data/README.md, Option A."
        )

    adata_sub = subsample_luca(
        source_h5ad=luca_core_path,
        n_target=100_000,
        seed=42,
    )
    print(adata_sub)
    save_adata(adata_sub, DEFAULT_SUBSAMPLE_NAME)
    print(f"\nWrote {sub_path} ({sub_path.stat().st_size / 1e6:.0f} MB)")
    del adata_sub  # release before reload

Building subsample from: D:\singleCell_rnaSeq\dataset\luca_core.h5ad


C:\Users\osmanjan\miniconda3\envs\scrnaseq\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
C:\Users\osmanjan\miniconda3\envs\scrnaseq\Lib\functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


AnnData object with n_obs × n_vars = 92452 × 17764
    obs: 'study', 'dataset', 'disease', 'sample', 'donor_id', 'cell_type', 'cell_type_major', 'ann_coarse', 'ann_fine', 'ever_smoker', 'sex', 'age', 'tumor_stage', 'uicc_stage', 'origin', 'origin_fine', 'platform', 'assay', 'n_genes_by_counts', 'total_counts', 'pct_counts_mito', 'total_counts_mito', 'doublet_status', 'EGFR_mutation', 'KRAS_mutation', 'TP53_mutation', 'ALK_mutation', 'BRAF_mutation'
    var: 'feature_biotype', 'feature_is_filtered', 'feature_length', 'feature_name', 'feature_reference', 'feature_type', 'is_highly_variable', 'mean_counts', 'mito', 'n_cells_by_counts', 'pct_dropout_by_counts', 'total_counts'
    layers: 'normalized_atlas'

Wrote E:\jobRelated\withClaude\resume_building\github-portfolio\scrnaseq-tumor-microenvironment\data\luca_subsample_100k.h5ad (774 MB)


## 2. Load the subsample and inspect cohort

In [3]:
adata = load_adata(DEFAULT_SUBSAMPLE_NAME)
print(adata)

print(f"\n--- cohort ---")
print(f"cells:    {adata.n_obs:,}")
print(f"genes:    {adata.n_vars:,}")
print(f"studies:  {adata.obs['study'].nunique()}")
print(f"diseases: {dict(adata.obs['disease'].value_counts())}")
print(f"\never_smoker:")
print(adata.obs['ever_smoker'].value_counts(dropna=False).to_string())

AnnData object with n_obs × n_vars = 92452 × 17764
    obs: 'study', 'dataset', 'disease', 'sample', 'donor_id', 'cell_type', 'cell_type_major', 'ann_coarse', 'ann_fine', 'ever_smoker', 'sex', 'age', 'tumor_stage', 'uicc_stage', 'origin', 'origin_fine', 'platform', 'assay', 'n_genes_by_counts', 'total_counts', 'pct_counts_mito', 'total_counts_mito', 'doublet_status', 'EGFR_mutation', 'KRAS_mutation', 'TP53_mutation', 'ALK_mutation', 'BRAF_mutation'
    var: 'feature_biotype', 'feature_is_filtered', 'feature_length', 'feature_name', 'feature_reference', 'feature_type', 'is_highly_variable', 'mean_counts', 'mito', 'n_cells_by_counts', 'pct_dropout_by_counts', 'total_counts'
    layers: 'normalized_atlas'

--- cohort ---
cells:    92,452
genes:    17,764
studies:  19
diseases: {'lung adenocarcinoma': np.int64(36663), 'normal': np.int64(23331), 'squamous cell lung carcinoma': np.int64(21337), 'non-small cell lung carcinoma': np.int64(7788), 'chronic obstructive pulmonary disease': np.int64

In [4]:
# Sanity check that .X is raw counts (sums should track obs['total_counts'])
sample_sums = np.asarray(adata.X[:5].sum(axis=1)).flatten()
print("Row sums of .X[:5] vs obs.total_counts[:5]:")
print(f"  .X row sums: {sample_sums.round(0)}")
print(f"  obs values:  {adata.obs['total_counts'].head().values.round(0)}")
# These should match closely (~5% gap is expected — obs.total_counts was
# computed on the unfiltered gene set; we keep 17,764 highly expressed genes).

Row sums of .X[:5] vs obs.total_counts[:5]:
  .X row sums: [ 1164.  3169. 13491.  2489.  5289.]
  obs values:  [ 1236.  3222. 13738.  2518.  5510.]


## 3. QC metrics — visualize what LuCA already computed

The atlas authors ran per-cell QC across all 19 source cohorts, so
`n_genes_by_counts`, `total_counts`, `pct_counts_mito`, and `doublet_status`
are already in `adata.obs`. We don't recompute — we visualize and pick filter
thresholds appropriate for our analysis (a stricter pass than the atlas's, to
keep the downstream classifier clean).

In [5]:
print(adata.obs[
    ["n_genes_by_counts", "total_counts", "pct_counts_mito"]
].describe().round(1))

       n_genes_by_counts  total_counts  pct_counts_mito
count            92452.0       92452.0          92452.0
mean              1517.6      245832.0              0.0
std               1083.0     1169956.8              0.0
min                182.0         395.0              0.0
25%                712.0        1602.0              0.0
50%               1170.0        3576.0              0.0
75%               2053.0       10213.2              0.0
max               9896.0    52479800.0              0.0


In [7]:
sc.pl.violin(
    adata,
    keys=["n_genes_by_counts", "total_counts", "pct_counts_mito"],
    jitter=0.4, multi_panel=True, show=False,
    save="_pre_filter.png",
)
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mito",
              show=False, save="_counts_vs_mt.png")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts",
              show=False, save="_counts_vs_genes.png")

<Axes: xlabel='total_counts', ylabel='n_genes_by_counts'>

## 4. Filter low-quality cells + drop doublets

Thresholds (review the violin plots above before changing them):

- `n_genes_by_counts ∈ [200, 6000]` — lower bound drops empty droplets; upper
  bound drops suspected multiplets.
- `pct_counts_mito ≤ 20` — drops dying / stressed cells.
- `doublet_status == 'singlet'` — uses LuCA's pre-computed doublet labels.

For LuCA's pre-cleaned subsample, retention is typically >99% — the heavy
lifting was already done upstream. The point of this step isn't to discover
new bad cells; it's to ensure our analytic frame matches the atlas's own
quality bar so the downstream results are honest.

In [8]:
MIN_GENES = 200
MAX_GENES = 6000
MAX_PCT_MT = 20.0

before = adata.n_obs
keep = (
    (adata.obs["n_genes_by_counts"] >= MIN_GENES)
    & (adata.obs["n_genes_by_counts"] <= MAX_GENES)
    & (adata.obs["pct_counts_mito"] <= MAX_PCT_MT)
)
if "doublet_status" in adata.obs:
    is_singlet = adata.obs["doublet_status"].astype(str).str.lower().isin(
        ["singlet", "false", "0", "none", "nan"]
    )
    if is_singlet.sum() > 0.5 * len(is_singlet):
        keep = keep & is_singlet
        print(f"applying doublet filter: {is_singlet.sum():,} singlets in subsample")

adata = adata[keep].copy()
print(f"\nBefore: {before:,} cells")
print(f"After:  {adata.n_obs:,} cells ({adata.n_obs/before:.1%} retained)")
print(f"\nPer-study counts (post-filter):")
print(adata.obs['study'].value_counts().to_string())

applying doublet filter: 92,452 singlets in subsample

Before: 92,452 cells
After:  92,013 cells (99.5% retained)

Per-study counts (post-filter):
study
Lambrechts_Thienpont_2018    9997
Wu_Zhou_2021                 7788
Goveia_Carmeliet_2020        6666
UKIM-V                       6664
Adams_Kaminski_2020          6664
Zilionis_Klein_2019          6656
Guo_Zhang_2018               5241
Maier_Merad_2020             4935
Maynard_Bivona_2020          4080
Mayr_Schiller_2020           3333
He_Fan_2021                  3333
Vieira_Teichmann_2019        3333
Madissoon_Meyer_2020         3333
Kim_Lee_2020                 3333
Laughney_Massague_2020       3333
Reyfman_Misharin_2018        3333
Chen_Zhang_2020              3332
Habermann_Kropski_2020       3331
Travaglini_Krasnow_2020      3328


## 5. Normalize, log1p, flag HVGs

Standard scanpy normalization: scale total counts per cell to 10,000, then
log1p. scVI in notebook 02 will use raw counts (preserved in
`layers["counts"]`) rather than the log-normalized values, but the log values
are needed for the within-this-notebook sanity-clustering and for marker-gene
scoring in notebook 03.

We use `flavor="seurat"` for HVG selection — equivalent to `seurat_v3` for our
purposes but doesn't require `scikit-misc`, which is fiddly to install on
Windows.

In [9]:
# Preserve raw counts (input format for scVI in notebook 02)
adata.layers["counts"] = adata.X.copy()

# Normalize + log1p — exactly once, on raw counts
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# HVG flagging
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")
print(f"HVGs flagged: {int(adata.var['highly_variable'].sum())}")

# What layers do we have now?
print(f"\nLayers: {list(adata.layers.keys())}")
print("  .X                          → log-normalized (this notebook + 03/04)")
print("  layers['counts']            → raw counts (notebook 02 scVI input)")
print("  layers['normalized_atlas']  → LuCA's published normalized values")

normalizing counts per cell
    finished (0:00:02)
extracting highly variable genes
    finished (0:00:02)
HVGs flagged: 2000

Layers: ['normalized_atlas', 'counts']
  .X                          → log-normalized (this notebook + 03/04)
  layers['counts']            → raw counts (notebook 02 scVI input)
  layers['normalized_atlas']  → LuCA's published normalized values


## 6. Save the filtered + normalized AnnData

Notebook 02 picks up from `data/processed/luca_filtered.h5ad`.

In [10]:
out_path = save_adata(adata, "processed/luca_filtered.h5ad")
print(f"Wrote {out_path}")
print(f"  size: {out_path.stat().st_size / 1e6:.0f} MB")

Wrote E:\jobRelated\withClaude\resume_building\github-portfolio\scrnaseq-tumor-microenvironment\data\processed\luca_filtered.h5ad
  size: 1138 MB


## Summary — Phase 1 complete

- Subsampled the Salcher 2022 LuCA atlas: ~892K → 92,452 cells balanced
  across 19 studies × 5 disease labels (seed=42).
- Confirmed `.X` carries raw counts by sliding `/layers/count` directly out of
  the source HDF5 (the atlas's own `.X` is pre-normalized, which is wrong
  input for scVI).
- Filtered using LuCA's pre-computed QC + `doublet_status` (>99% retention —
  the atlas was already clean).
- Normalized, log1p'd, flagged 2000 HVGs.
- Wrote `data/processed/luca_filtered.h5ad` (~1.1 GB).

**Next:** [`02_scvi_integration.ipynb`](02_scvi_integration.ipynb) — train
scVI with `study` as the batch key, evaluate batch-mixing on the latent
embedding, and save a checkpoint for downstream notebooks.